In [30]:
#import libraries 
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [31]:
#import data
with open("text_dataset0.txt", "r", encoding="utf-8") as f:
    new_data = f.read()

print(new_data[:5000])

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.

As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.

In the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.

As the stars began to twinkle in the night sky, the crowd grew even larger, and the festivities continued well into the night. It was a day filled with joy, laughter, and memories that would last a lifetime.


The ancient

In [32]:
#propcess text
tokenizer = Tokenizer(num_words=8000, oov_token="<OOV>")
tokenizer.fit_on_texts([new_data])

total_words = len(tokenizer.word_index) + 1
print(total_words)

11598


In [33]:
import pickle

# save tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)


In [34]:
#sequence of tokens

input_sequences = []

for line in new_data.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])


input_sequences = input_sequences[:8000]

print(len(tokenizer.word_index) + 1)


8931


In [35]:
print("Total sequences:", len(input_sequences))

Total sequences: 8000


In [36]:
#padding 
max_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [37]:
#data spliting
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [38]:
#build the LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

vocab_size = len(tokenizer.word_index) + 1

model = Sequential()
model.add(Embedding(vocab_size, 100, input_length=max_len-1))
model.add(LSTM(150))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])



In [39]:
X = np.array(X)
y = np.array(y).reshape(-1)

print("X:", X.shape)
print("y:", y.shape)

X: (8000, 136)
y: (8000,)


In [40]:
print("Max value in X:", X.max())
print("Max value in y:", y.argmax().max())

Max value in X: 4998
Max value in y: 2004


In [41]:
# from tensorflow.keras.models import load_model
# model = load_model("sugg_next_word_v2.h5")

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam',  metrics=['accuracy'])

In [42]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.fit(X, y, epochs=20, batch_size=128)

Epoch 1/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 42s 559ms/step - accuracy: 0.0675 - loss: 7.0088
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 42s 671ms/step - accuracy: 0.0891 - loss: 5.7578
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 571ms/step - accuracy: 0.0891 - loss: 5.6780
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 37s 586ms/step - accuracy: 0.0891 - loss: 5.6173
Epoch 5/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 35s 554ms/step - accuracy: 0.0959 - loss: 5.5618
Epoch 6/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 565ms/step - accuracy: 0.0997 - loss: 5.5131
Epoch 7/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 578ms/step - accuracy: 0.0994 - loss: 5.4744
Epoch 8/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 32s 513ms/step - accuracy: 0.1031 - loss: 5.4254
Epoch 9/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 35s 551ms/step - accuracy: 0.1190 - loss: 5.3644
Epoch 10/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 34s 536ms/step - accuracy: 0.1258 - loss: 5.2940
Epoch 11/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 579ms/step - accuracy: 0.1340 - loss: 5.2244
Epoch 12/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 42

In [44]:
model.save("sugg_next_word_v3.h5")

In [45]:
# #function for prediction 
# def predict_next_words(text, n_words=3):
#     for _ in range(n_words):
#         token_list = tokenizer.texts_to_sequences([text])[0]
#         token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        
#         predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        
#         output_word = ""
#         for word, index in tokenizer.word_index.items():
#             if index == predicted:
#                 output_word = word
#                 break
        
#         text += " " + output_word
    
#     return text

In [46]:
# testing
# print(predict_next_words("the sun was", 10))

In [47]:
def predict_top_words(text, n_words=3):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predictions = model.predict(token_list, verbose=0)[0]
    top_indices = predictions.argsort()[-n_words:][::-1]
    
    index_to_word = {index: word for word, index in tokenizer.word_index.items()}
    return [index_to_word.get(i, "") for i in top_indices]

In [48]:
def interactive_prediction():
    text = input("Enter your sentence: ")
    
    while True:
        suggestions = predict_top_words(text)
        
        print("\nSuggestions:")
        for i, word in enumerate(suggestions):
            print(f"{i+1}. {word}")
        
        choice = input("Choose word (1/2/3) or 0 for exit: ")
        
        if choice == '0':
            break
        
        if choice in ['1', '2', '3']:
            selected_word = suggestions[int(choice)-1]
            text += " " + selected_word
            print("Updated sentence:", text)
        else:
            print("Invalid choice")
    
    print("\nFinal sentence:", text)
    
interactive_prediction()



Suggestions:
1. a
2. to
3. as
Updated sentence: the girl is to

Suggestions:
1. the
2. and
3. a

Final sentence: the girl is to
